# X1: Debugging Deep Networks

Every earlier lesson assumed training just worked. In practice it usually
doesn't, the first few times — and it fails in a small number of
recognisable ways, independent of which architecture from lessons 0-15 is
involved. This notebook is a toolkit: a taxonomy of loss-curve pathologies,
the single most useful sanity check before trusting any result, and how to
locate the layer or step where a training run actually broke.

## Introduction

A model that trains to a good result and a model that silently fails
produce code that looks identical and often even runs without an error.
The difference shows up only in things practitioners learn to check:
the shape of the loss curve, whether a trivially small problem can be
solved at all, whether any unit in the network is still receiving
gradient, and whether the same run produces the same answer twice. Every
technique here is architecture-agnostic — it applies exactly the same way
whether the network in question is 2a's MLP or 10a's Transformer.

## Setup

In [ ]:
# Fixed seeds: every stochastic step in this notebook (weight init, minibatch
# order, deliberately injected failures) is reproducible.
import numpy as np
import torch

SEED = 0
np.random.seed(SEED)
torch.manual_seed(SEED)

import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt

plt.rcParams["figure.figsize"] = (5, 4)
print("numpy:", np.__version__)
print("torch:", torch.__version__)

device = torch.device("cpu")
print("using device:", device)

In [ ]:
# A small synthetic classification problem is enough for every demo below --
# the failure modes shown are properties of training dynamics, not of any
# particular dataset.
def make_classification_data(n, seed):
    g = torch.Generator().manual_seed(seed)
    X = torch.randn(n, 20, generator=g)
    true_w = torch.randn(20, 4, generator=g)
    logits = X @ true_w
    y = logits.argmax(dim=1)
    return X, y


X_train, y_train = make_classification_data(512, SEED)
X_small, y_small = X_train[:8], y_train[:8]  # one small batch, reused throughout
print("X_train:", X_train.shape, " classes:", y_train.unique().tolist())

## Reading Loss Curves

Most training failures are diagnosable from the loss curve's *shape* alone,
before looking at anything else:

| Curve shape | Usual cause |
|---|---|
| Smooth, monotonic decrease | Healthy |
| Flat from step one, never moves | Learning rate far too low, or gradients not reaching the parameters at all |
| Explodes to very large values or NaN | Learning rate far too high |
| Decreases then diverges partway through | Learning rate borderline-too-high; instability compounds over time |
| Train loss falls, validation loss rises | Overfitting |
| Falls slowly then plateaus early, well above zero | Underfitting -- insufficient capacity or training time |

The three learning-rate-driven shapes are directly comparable on the exact
same model and data, changing only the learning rate.

In [ ]:
def make_model(seed=SEED):
    torch.manual_seed(seed)
    return nn.Sequential(nn.Linear(20, 64), nn.ReLU(), nn.Linear(64, 4))


def train_curve(lr, steps=60, clamp=1e6):
    model = make_model()
    opt = optim.SGD(model.parameters(), lr=lr)
    losses = []
    for _ in range(steps):
        loss = nn.functional.cross_entropy(model(X_train), y_train)
        losses.append(min(loss.item(), clamp) if not torch.isnan(loss) else float("nan"))
        if torch.isnan(loss) or loss.item() > clamp:
            break
        opt.zero_grad(); loss.backward(); opt.step()
    return losses


curves = {"lr=0.001 (too low)": train_curve(0.001), "lr=0.1 (healthy)": train_curve(0.1),
          "lr=20 (too high)": train_curve(20.0)}

fig, ax = plt.subplots(figsize=(7, 4.5))
for label, losses in curves.items():
    ax.plot(losses, label=label, marker=".")
ax.set_xlabel("step"); ax.set_ylabel("loss"); ax.set_yscale("log"); ax.legend()
ax.set_title("Same model and data, three learning rates")
plt.show()
for label, losses in curves.items():
    print(f"{label}: final loss = {losses[-1]:.3g}, {len(losses)} steps completed")

## Overfit One Batch

Before trusting *any* result on the full dataset, check whether the model
can drive loss to near zero on a single small batch. Real data, real
labels, deliberately no regularisation, as many steps as needed. A correct
training loop, correct loss, and correctly-wired optimiser should solve
this trivially. **If it can't, nothing downstream is worth debugging until
this does** -- generalisation failures are meaningless to chase on top of a
pipeline that can't even memorise eight examples.

In [ ]:
def overfit_one_batch(model, optimizer, steps=200):
    losses = []
    for _ in range(steps):
        loss = nn.functional.cross_entropy(model(X_small), y_small)
        losses.append(loss.item())
        optimizer.zero_grad(); loss.backward(); optimizer.step()
    return losses


# The correct setup: the optimizer holds exactly the parameters used in the forward pass.
correct_model = make_model()
correct_losses = overfit_one_batch(correct_model, optim.Adam(correct_model.parameters(), lr=0.1))

# A real, common bug: the optimizer was built over a *different* model instance
# than the one actually used in the forward pass -- e.g. a leftover reference
# from refactoring, or a model rebuilt after the optimizer was constructed.
used_model = make_model()
decoy_model = make_model()  # never called in the forward pass below
broken_opt = optim.Adam(decoy_model.parameters(), lr=0.1)
broken_losses = []
for _ in range(200):
    loss = nn.functional.cross_entropy(used_model(X_small), y_small)
    broken_losses.append(loss.item())
    broken_opt.zero_grad(); loss.backward(); broken_opt.step()

fig, ax = plt.subplots(figsize=(7, 4.5))
ax.plot(correct_losses, label="correct: optimizer wired to the model actually used")
ax.plot(broken_losses, label="broken: optimizer wired to the wrong model instance")
ax.set_xlabel("step"); ax.set_ylabel("loss on the 8-example batch"); ax.legend()
ax.set_title("The overfit-one-batch test")
plt.show()
print(f"correct setup -- final loss: {correct_losses[-1]:.4f}")
print(f"broken setup  -- final loss: {broken_losses[-1]:.4f}  (stuck near chance level)")
assert correct_losses[-1] < 0.01, "the correct setup should overfit a single batch to near zero"
assert broken_losses[-1] > 1.0, "the broken setup should fail to overfit -- gradients update the wrong object" 

## Gradient and Activation Diagnostics

Two cheap checks localise *where* a stalled network is broken, once the
loss curve or the overfit-one-batch test has flagged that something is
wrong.

**Dead ReLUs**: a ReLU unit whose pre-activation is negative for every
input in a batch outputs exactly zero for all of them, and its gradient is
then zero too -- it stops learning permanently ('dying ReLU'), most often
from a too-large negative bias or a too-large learning rate pushing weights
into that regime. Counting units with zero activation across a whole batch
finds this directly.

In [ ]:
def fraction_dead_relu(model, X):
    with torch.no_grad():
        activations = model[1](model[0](X))  # output of the first ReLU
    return (activations.sum(dim=0) == 0).float().mean().item()


healthy_model = make_model()
broken_init_model = make_model()
with torch.no_grad():
    broken_init_model[0].bias.fill_(-10.0)  # deliberately pushes every pre-activation negative

print(f"fraction of dead units, default init:        {fraction_dead_relu(healthy_model, X_train):.2f}")
print(f"fraction of dead units, bad negative-bias init: {fraction_dead_relu(broken_init_model, X_train):.2f}")

**Per-layer gradient norms**: 3a and 6a both showed vanishing gradients as
a *depth* problem -- the earliest layers of a deep plain stack receive a
vanishingly small gradient magnitude. The same per-layer-norm check that
proved that theoretically is exactly what to run on a real stalled model.

In [ ]:
def per_layer_grad_norms(model, X, y):
    model.zero_grad()
    loss = nn.functional.cross_entropy(model(X), y)
    loss.backward()
    return [p.grad.norm().item() for p in model.parameters() if p.grad is not None and p.dim() == 2]


torch.manual_seed(SEED)
deep_plain = nn.Sequential(*[layer for _ in range(8) for layer in (nn.Linear(20, 20), nn.ReLU())][:-1] + [nn.Linear(20, 4)])
X20, y20 = make_classification_data(64, SEED)
norms = per_layer_grad_norms(deep_plain, X20, y20)

fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(range(len(norms)), norms)
ax.set_xlabel("layer index (input -> output)"); ax.set_ylabel("gradient norm"); ax.set_yscale("log")
ax.set_title("Per-layer gradient norms, an 8-layer plain stack")
plt.show()
print("earliest-layer norm:", f"{norms[0]:.2e}", " latest-layer norm:", f"{norms[-1]:.2e}")

## Finding NaNs

Once a NaN appears, everything downstream is NaN too -- the useful
question is *where it first appeared*, not where it was noticed. Checking
the loss at every step, rather than only printing it every few hundred, is
the cheapest way to localise the exact step; the usual causes are a
learning rate large enough to overflow a weight to `inf` (whose gradient is
then NaN), `log(0)` inside a loss that lacks numerical-stability guards, and
division by a quantity that can legitimately be zero (an empty batch bucket,
a normalisation constant computed from an all-zero row).

In [ ]:
torch.manual_seed(SEED)
nan_model = make_model()
nan_opt = optim.SGD(nan_model.parameters(), lr=1000.0)  # deliberately far too large

first_nan_step = None
for step in range(30):
    loss = nn.functional.cross_entropy(nan_model(X_train), y_train)
    if torch.isnan(loss) or torch.isinf(loss):
        first_nan_step = step
        break
    nan_opt.zero_grad(); loss.backward(); nan_opt.step()

print(f"first NaN loss detected at step {first_nan_step}")
print("largest weight magnitude just before the NaN step:",
      max(p.abs().max().item() for p in nan_model.parameters()))
assert first_nan_step is not None, "the deliberately-too-large learning rate should produce a NaN" 

## Reproducibility and Seeds

`np.random.seed` and `torch.manual_seed`, set once at the top of every
notebook in this series, control weight initialisation, minibatch shuffling
order, and any explicit noise sampling (dropout masks, the reparameterisation
trick's $\epsilon$). Two runs with every relevant seed fixed should be
bit-for-bit identical.

In [ ]:
def run_and_hash(seed):
    torch.manual_seed(seed)
    model = make_model(seed=seed)
    opt = optim.SGD(model.parameters(), lr=0.1)
    for _ in range(10):
        loss = nn.functional.cross_entropy(model(X_train), y_train)
        opt.zero_grad(); loss.backward(); opt.step()
    return loss.item()


run_a = run_and_hash(SEED)
run_b = run_and_hash(SEED)
print(f"run A final loss: {run_a!r}")
print(f"run B final loss: {run_b!r}")
assert run_a == run_b, "identical seeds should produce a bit-for-bit identical result" 

Seeding does not make everything deterministic, though -- three common gaps
remain even with every seed fixed:

- **GPU non-determinism**: several cuDNN convolution and pooling algorithms
  pick a fast, non-deterministic implementation by default; `torch.use_
  deterministic_algorithms(True)` forces deterministic ones (usually
  slower) or raises an error naming the offending op.
- **DataLoader workers**: multiple worker processes fetch batches out of
  order unless the loader's own generator is separately seeded per worker,
  independent of the main process's seed.
- **Floating-point summation order**: parallel reductions across threads
  can sum the same values in a different order, and floating-point addition
  is not associative -- a difference visible only in the last few bits,
  but a real one.

The practical rule: seeding the model and the data order is necessary but
not sufficient for full determinism -- state it, and know which of the
three gaps above applies to the current setup.

## Key Takeaways

- A loss curve's **shape alone** narrows the cause before touching anything
  else -- flat means the learning rate or gradient flow is broken, diverging
  means the learning rate is too high, train/val divergence means
  overfitting.
- **Overfit one batch first.** A setup that can't drive loss to near zero on
  eight examples has a structural bug -- confirmed here by an optimizer
  wired to the wrong model instance, a realistic mistake that a full-dataset
  run would have made far harder to spot.
- **Dead ReLUs** (zero activation across a whole batch) and **per-layer
  gradient norms** localise a stall to a specific layer or initialisation
  choice, rather than leaving it as a single unexplained flat loss curve.
- **NaN losses have a first step.** Checking every step, not just a
  periodic printout, finds it -- and the usual causes are a learning rate
  large enough to overflow, or an unguarded `log(0)` or division by zero.
- **Seeding is necessary but not sufficient** for reproducibility -- GPU
  algorithm choice, DataLoader worker seeding, and floating-point summation
  order can all remain non-deterministic even with the model and data order
  fully seeded.